In [1]:
%matplotlib inline

In [2]:
#Import your libraries here

import numpy as np
import pandas as pd
import scipy.optimize as sco
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import itertools
import os
import sys
from pathlib import Path

In [3]:
notebook_path = Path(os.getcwd()).resolve()

def get_root(path):
    for parent in [path] + list(path.parents):
        if (parent / "static_data").exists():
            return parent
    return path

PROJECT_ROOT = get_root(notebook_path)
DATA_DIR = PROJECT_ROOT / "static_data"

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project Root: {PROJECT_ROOT}")
print(f"Data Source:  {DATA_DIR}")

Project Root: D:\Users\kamen.dimitrov\Desktop\SOFTUNI\AI_and_ML_upskill_program\Data_Science\08_final_project
Data Source:  D:\Users\kamen.dimitrov\Desktop\SOFTUNI\AI_and_ML_upskill_program\Data_Science\08_final_project\static_data


In [4]:
import importlib

BASE_URL = "https://storage.googleapis.com/softuni_data_science_final_project_kamend1/static_data/"

from src.data_pipeline_utils import data_fetching_handling as data_pipe
import src.plotting_utils.plotting_utils as plot_utils
importlib.reload(plot_utils)
importlib.reload(data_pipe)

<module 'src.data_pipeline_utils.data_fetching_handling' from 'D:\\Users\\kamen.dimitrov\\Desktop\\SOFTUNI\\AI_and_ML_upskill_program\\Data_Science\\08_final_project\\src\\data_pipeline_utils\\data_fetching_handling.py'>

# #TODO

text description

In [5]:
tickers = [
    "AAPL", "GOOG", "MSFT", "NVDA",
    "JPM", "BAC", "F", "UPS", "WMT", "TGT",
    "VZ", "T", "FE", "PFE", "JNJ", "DIS",
    "V", "MCD", "NKE", "XOM", "CVX",
    "CAT", "DE", "LMT", "AMD", "INTC", "ORCL", 
    "CRM", "CB", "PG"
]
returns_df = pd.read_csv(BASE_URL + "stock_returns_static_dataset.csv")
returns_df["Date"] = pd.to_datetime(returns_df["Date"])

test_returns_df = returns_df[returns_df["Date"] <= "2021-03-31"].copy()
test_returns_df = test_returns_df.set_index("Date").sort_index()

target_returns_df = returns_df[returns_df["Date"] >= "2021-04-01"].copy()
target_returns_df = target_returns_df.set_index("Date").sort_index()

print(test_returns_df.shape, target_returns_df.shape)
print(test_returns_df.index.min(), test_returns_df.index.max())
print(target_returns_df.index.min(), target_returns_df.index.max())

(1259, 30) (1254, 30)
2016-04-01 00:00:00 2021-03-31 00:00:00
2021-04-01 00:00:00 2026-03-30 00:00:00


In [6]:
trading_days = 252
risk_free_rate = 0.03
mean_returns = test_returns_df.mean() * trading_days
cov_matrix = test_returns_df.cov() * trading_days

In [7]:
number_of_stocks = len(tickers)
weights = np.array([1 / number_of_stocks] * number_of_stocks)

equal_weight_portfolio = list(zip(tickers, weights))

returns = np.dot(weights, mean_returns)
std_dev = np.sqrt(np.dot(weights.T, np.dot(cov_matrix, weights)))
print(returns, std_dev)

0.18086381875792062 0.19331943148184835


In [8]:
args = (mean_returns, cov_matrix)

# Constraint: sum of weights equals 1
constraints = ({'type': 'eq', 'fun': lambda x: np.sum(x) - 1})

# Bounds: weights between 0 and 1
bounds = tuple((0, 1) for asset in range(number_of_stocks))

# Initial guess (equal distribution)
initial_guess = number_of_stocks * [1. / number_of_stocks]

In [9]:
def negative_sharpe_ratio(weights, mean_returns, cov_matrix, risk_free_rate=0.03):
    p_ret = np.dot(weights, mean_returns)
    p_std = np.sqrt(np.dot(weights.T, np.dot(cov_matrix, weights)))
    return -(p_ret - risk_free_rate) / p_std

In [10]:
optimal_portfolio = sco.minimize(
    negative_sharpe_ratio, 
    initial_guess, 
    args=args,
    method='SLSQP', 
    bounds=bounds, 
    constraints=constraints
)

optimal_weights = optimal_portfolio.x
optimal_return = np.sum(mean_returns * optimal_weights)
optimal_variance = np.dot(optimal_weights.T, np.dot(cov_matrix, optimal_weights))
optimal_volatility = np.sqrt(optimal_variance)
optimal_sharpe_ratio = optimal_return / optimal_volatility

print(f"Expected Return: {optimal_return:.4f}")
print(f"Variance: {optimal_variance:.4f}")

optimal_portfolio_metrics = {"Expected Return": [optimal_return],
                               "Volatility": [optimal_volatility],}

pd.DataFrame(optimal_portfolio_metrics) \
    .to_csv(DATA_DIR / "optimal_portfolio_metrics.csv", index=False)

Expected Return: 0.3998
Variance: 0.0763


In [11]:
optimal_portfolio = list(zip(tickers, optimal_weights))

print(optimal_portfolio, equal_weight_portfolio)

[('AAPL', np.float64(0.04837764271223956)), ('GOOG', np.float64(0.0)), ('MSFT', np.float64(0.18354340437114372)), ('NVDA', np.float64(0.1516085623008635)), ('JPM', np.float64(3.465450861721707e-16)), ('BAC', np.float64(0.0)), ('F', np.float64(2.6584109238107203e-17)), ('UPS', np.float64(1.1538446381929522e-16)), ('WMT', np.float64(0.08002236243342954)), ('TGT', np.float64(0.03757356505353604)), ('VZ', np.float64(0.0)), ('T', np.float64(0.0)), ('FE', np.float64(4.145963980101059e-18)), ('PFE', np.float64(0.0)), ('JNJ', np.float64(0.0)), ('DIS', np.float64(0.0)), ('V', np.float64(0.0)), ('MCD', np.float64(0.0)), ('NKE', np.float64(0.0)), ('XOM', np.float64(1.260359961695797e-16)), ('CVX', np.float64(0.0)), ('CAT', np.float64(6.39550701184002e-18)), ('DE', np.float64(0.32387328977945407)), ('LMT', np.float64(2.8918834903068325e-17)), ('AMD', np.float64(0.1750011733493341)), ('INTC', np.float64(0.0)), ('ORCL', np.float64(1.90276372479333e-16)), ('CRM', np.float64(2.0493077483884108e-16)), 

In [12]:
def calculate_rebalanced_buy_and_hold(
    df,
    weights,
    tickers,
    initial_capital=1_000_000
):

    returns_df = df[tickers].copy()

     # only keep weights for the same columns and same order
    weights = weights[tickers]
    weights = weights / weights.sum()
    
    portfolio_returns = df.dot(weights)

    portfolio_value = initial_capital * (1 + portfolio_returns).cumprod()

    ending_value = portfolio_value.iloc[-1]

    return portfolio_returns, portfolio_value, ending_value

In [14]:
optimal_weights = pd.Series(dict(optimal_portfolio))
equal_weights = pd.Series(dict(equal_weight_portfolio))

opt_ret, opt_value, opt_end = calculate_rebalanced_buy_and_hold(
    target_returns_df,
    optimal_weights,
    tickers,
    initial_capital=1_000_000
)

eq_ret, eq_value, eq_end = calculate_rebalanced_buy_and_hold(
    target_returns_df,
    equal_weights,
    tickers,
    initial_capital=1_000_000
)

print("Optimal portfolio ending value:", round(opt_end, 2))
print("Equal weight portfolio ending value:", round(eq_end, 2))

Optimal portfolio ending value: 2068535.33
Equal weight portfolio ending value: 1503203.18


In [16]:
final_signal_export_df = pd.read_csv(BASE_URL + "final_signal_export.csv")

#Alternative reproduction path
#final_signal_export_df = pd.read_csv(DATA_DIR / "final_signal_export.csv")

final_signal_export_df

,Unnamed: 0,Date,Ticker,open_price,close_price,combined_signal
0,0,2016-04-01,AAPL,24.636544,24.684103,1
1,1,2016-04-04,AAPL,25.007968,24.910585,1
2,2,2016-04-05,AAPL,24.801878,25.166506,1
3,3,2016-04-06,AAPL,24.964938,24.869822,1
4,4,2016-04-07,AAPL,24.901522,25.130268,1
...,...,...,...,...,...,...
75205,75205,2026-03-16,XOM,156.000000,156.119995,0
75206,75206,2026-03-17,XOM,158.250000,157.229996,0
75207,75207,2026-03-18,XOM,159.660004,158.809998,0
75208,75208,2026-03-19,XOM,158.259995,157.589996,0


In [17]:
final_signal_export_df['combined_signal'].value_counts()

combined_signal
 0    70169
-1     3224
 1     1817
Name: count, dtype: int64

In [53]:
def backtest_signal_strategy(
    df,
    initial_capital=1_000_000,
    annual_cash_rate=0.03,
    buy_cost_multiplier=1.01,
    sell_cost_multiplier=0.99,
    max_position_weight=0.15,
    min_trade_dollars=1000
):
    cash = initial_capital
    positions = {}

    daily_records = []
    trade_log = []

    daily_rate = annual_cash_rate / 365
    unique_dates = df["Date"].sort_values().unique()

    for current_date in unique_dates:
        day_data = df[df["Date"] == current_date].copy()

        close_prices = dict(zip(day_data["Ticker"], day_data["close_price"]))
        prev_close_prices = dict(zip(day_data["Ticker"], day_data["prev_close"]))

        holdings_value_before = 0
        for ticker, shares in positions.items():
            if ticker in prev_close_prices and pd.notna(prev_close_prices[ticker]):
                holdings_value_before += shares * prev_close_prices[ticker]

        portfolio_value_before = cash + holdings_value_before

        # count active buy signals for the day
        active_buy_count = (day_data["prev_signal"] == 1).sum()

        if active_buy_count > 0:
            dynamic_target_weight = min(1 / active_buy_count, max_position_weight)
        else:
            dynamic_target_weight = 0

        for _, row in day_data.iterrows():
            ticker = row["Ticker"]
            prev_signal = row["prev_signal"]
            open_price = row["open_price"]

            if pd.isna(prev_signal):
                continue

            current_shares = positions.get(ticker, 0.0)
            current_position_value = current_shares * open_price

            # SELL: liquidate fully
            if prev_signal == -1:
                if current_shares > 0:
                    sell_price = open_price * sell_cost_multiplier
                    proceeds = current_shares * sell_price
                    cash += proceeds

                    trade_log.append({
                        "Date": current_date,
                        "Ticker": ticker,
                        "Action": "SELL",
                        "SignalUsed": prev_signal,
                        "Shares": current_shares,
                        "TradePrice": sell_price,
                        "TradeValue": proceeds
                    })

                    positions[ticker] = 0.0

            # BUY / TOP-UP
            elif prev_signal == 1 and dynamic_target_weight > 0:
                target_position_value = portfolio_value_before * dynamic_target_weight
                gap_to_target = target_position_value - current_position_value

                if gap_to_target > 0:
                    buy_price = open_price * buy_cost_multiplier
                    amount_to_invest = min(gap_to_target, cash)

                    if amount_to_invest >= min_trade_dollars:
                        shares_to_buy = amount_to_invest / buy_price

                        cash -= amount_to_invest
                        positions[ticker] = current_shares + shares_to_buy

                        trade_log.append({
                            "Date": current_date,
                            "Ticker": ticker,
                            "Action": "BUY",
                            "SignalUsed": prev_signal,
                            "Shares": shares_to_buy,
                            "TradePrice": buy_price,
                            "TradeValue": amount_to_invest
                        })

        holdings_value_after = 0
        for ticker, shares in positions.items():
            if shares > 0 and ticker in close_prices:
                holdings_value_after += shares * close_prices[ticker]

        total_portfolio_value = cash + holdings_value_after

        beginning_cash = cash
        cash_interest = beginning_cash * daily_rate
        cash = beginning_cash + cash_interest

        daily_records.append({
            "Date": current_date,
            "BeginningCash": beginning_cash,
            "CashInterest": cash_interest,
            "Cash": cash,
            "HoldingsValue": holdings_value_after,
            "TotalPortfolioValue": total_portfolio_value,
            "NumOpenPositions": sum(1 for s in positions.values() if s > 0),
            "ActiveBuySignals": int(active_buy_count),
            "DynamicTargetWeight": dynamic_target_weight
        })

    daily_df = pd.DataFrame(daily_records)
    trade_log_df = pd.DataFrame(trade_log)
    daily_df["DailyReturn"] = daily_df["TotalPortfolioValue"].pct_change().fillna(0)

    last_day_data = df.sort_values("Date").groupby("Ticker").tail(1)
    last_close_prices = dict(zip(last_day_data["Ticker"], last_day_data["close_price"]))

    final_holdings_value = 0
    final_positions_valuation = []

    for ticker, shares in positions.items():
        if shares > 0 and ticker in last_close_prices:
            last_close = last_close_prices[ticker]
            market_value = shares * last_close
            final_holdings_value += market_value

            final_positions_valuation.append({
                "Ticker": ticker,
                "Shares": shares,
                "LastClose": last_close,
                "MarketValue": market_value
            })

    final_positions_df = pd.DataFrame(final_positions_valuation)

    cash_row = pd.DataFrame([{
        "Ticker": "CASH",
        "Shares": np.nan,
        "LastClose": 1.0,
        "MarketValue": cash
    }])

    final_positions_df = pd.concat([final_positions_df, cash_row], ignore_index=True)
    final_portfolio_value = final_positions_df["MarketValue"].sum()

    final_positions_df["Weight"] = (
        final_positions_df["MarketValue"] / final_portfolio_value
    )

    final_positions_df = final_positions_df.sort_values(
        "MarketValue", ascending=False
    ).reset_index(drop=True)

    return daily_df, trade_log_df, positions, final_positions_df, final_portfolio_value

In [54]:
df = final_signal_export_df.copy()
df["Date"] = pd.to_datetime(df["Date"])
df = df[df['Date'] >= '2021-04-01']
df = df.sort_values(["Ticker", "Date"])

df["prev_signal"] = df.groupby("Ticker")["combined_signal"].shift(1)
df["prev_close"] = df.groupby("Ticker")["close_price"].shift(1)

df = df.sort_values(["Date", "Ticker"]).reset_index(drop=True)

daily_df, trade_log_df, final_positions, final_positions_df, final_portfolio_value = backtest_signal_strategy(
    df,
    initial_capital=1_000_000,
    annual_cash_rate=0.03,
    buy_cost_multiplier=1.01,
    sell_cost_multiplier=0.99
)

print("Final portfolio value:", round(final_portfolio_value, 2))
print(final_positions_df.sort_values("MarketValue", ascending=False))

Final portfolio value: 1343843.03
  Ticker       Shares   LastClose    MarketValue    Weight
0   CASH          NaN    1.000000  552913.884252  0.411442
1   NVDA  1144.591878  178.559998  204378.323000  0.152085
2    DIS  2053.494388   99.199997  203706.637038  0.151585
3    BAC  4297.705370   47.009998  202035.122235  0.150341
4   MSFT   464.780895  389.019989  180809.058634  0.134546


In [55]:
trade_log_df

,Date,Ticker,Action,SignalUsed,Shares,TradePrice,TradeValue
0,2021-04-28,VZ,BUY,1.0,3545.461428,42.370246,150222.072915
1,2021-05-17,AMD,BUY,1.0,2015.698325,74.962201,151101.183497
2,2021-05-19,AMD,BUY,1.0,43.458830,73.891604,3211.242637
3,2021-06-01,VZ,BUY,1.0,35.812080,42.655869,1527.595407
4,2021-06-02,VZ,BUY,1.0,42.172679,42.325150,1784.964984
...,...,...,...,...,...,...,...
131,2026-01-21,MSFT,BUY,1.0,76.098924,456.085013,34707.578507
132,2026-02-03,T,SELL,-1.0,8593.584588,25.848901,222134.713861
133,2026-02-13,AAPL,SELL,-1.0,845.980788,259.389910,219438.880091
134,2026-03-17,DIS,BUY,1.0,605.057821,100.161698,60603.618813


In [56]:
total_commission = (trade_log_df["TradeValue"] * 0.01).sum()

print(f"Total transaction cost: ${total_commission:,.2f}")

Total transaction cost: $154,714.76


In [57]:
daily_df

,Date,BeginningCash,CashInterest,Cash,HoldingsValue,TotalPortfolioValue,NumOpenPositions,ActiveBuySignals,DynamicTargetWeight,DailyReturn
0,2021-04-01,1.000000e+06,82.191781,1.000082e+06,0.000000,1.000000e+06,0,0,0.00,0.000000
1,2021-04-05,1.000082e+06,82.198536,1.000164e+06,0.000000,1.000082e+06,0,0,0.00,0.000082
2,2021-04-06,1.000164e+06,82.205292,1.000247e+06,0.000000,1.000164e+06,0,0,0.00,0.000082
3,2021-04-07,1.000247e+06,82.212049,1.000329e+06,0.000000,1.000247e+06,0,0,0.00,0.000082
4,2021-04-08,1.000329e+06,82.218806,1.000411e+06,0.000000,1.000329e+06,0,0,0.00,0.000082
...,...,...,...,...,...,...,...,...,...,...
1243,2026-03-16,6.153306e+05,50.575120,6.153812e+05,732729.985544,1.348061e+06,4,0,0.00,-0.006011
1244,2026-03-17,5.547776e+05,45.598158,5.548232e+05,798431.074109,1.353209e+06,4,1,0.15,0.003819
1245,2026-03-18,5.548232e+05,45.601906,5.548688e+05,800983.224701,1.355806e+06,4,1,0.15,0.001920
1246,2026-03-19,5.528230e+05,45.437507,5.528684e+05,794000.837317,1.346824e+06,4,1,0.15,-0.006625


In [58]:
avg_cash = daily_df["Cash"].mean()

print(f"Average cash: ${avg_cash:,.2f}")

daily_df["CashWeight"] = daily_df["Cash"] / daily_df["TotalPortfolioValue"]

avg_cash_weight = daily_df["CashWeight"].mean()

print(f"Average cash weight: {avg_cash_weight:.2%}")

avg_invested_weight = 1 - avg_cash_weight

print(f"Average invested weight: {avg_invested_weight:.2%}")

print(daily_df["CashWeight"].describe())

Average cash: $366,685.59
Average cash weight: 32.70%
Average invested weight: 67.30%
count    1248.000000
mean        0.326964
std         0.350153
min         0.000000
25%         0.000000
50%         0.158012
75%         0.633642
max         1.000082
Name: CashWeight, dtype: float64


In [59]:
n_days = len(daily_df)
years = n_days / 252
initial_capital = 1_000_000

# 1) Annualized total return of the signal strategy
signal_total_return = final_portfolio_value / initial_capital - 1
signal_annual_return = (1 + signal_total_return) ** (1 / years) - 1

# 2) Cash interest earned in DOLLARS
total_cash_interest = daily_df["CashInterest"].sum()

# 3) Cash return over the full period
cash_total_return = total_cash_interest / initial_capital

# 4) Annualized cash return
cash_annual_return = (1 + cash_total_return) ** (1 / years) - 1

# 5) Annualized strategy return net of cash
strategy_annual_return_net_of_cash = signal_annual_return - cash_annual_return

# 6) Average market exposure
daily_df["Exposure"] = daily_df["HoldingsValue"] / daily_df["TotalPortfolioValue"]
avg_exposure = daily_df["Exposure"].mean()

# 7) Exposure-adjusted annual return net of cash
exposure_adjusted_annual_return_net_of_cash = (
    strategy_annual_return_net_of_cash / avg_exposure
)

# 8) Annualized returns of the other two portfolios
opt_total_return = opt_end / initial_capital - 1
opt_annual_return = (1 + opt_total_return) ** (1 / years) - 1

eq_total_return = eq_end / initial_capital - 1
eq_annual_return = (1 + eq_total_return) ** (1 / years) - 1

# 9) Print results
print(f"Final portfolio value: ${final_portfolio_value:,.2f}")
print(f"Signal strategy annual return: {signal_annual_return:.2%}")
print(f"Total cash interest earned: ${total_cash_interest:,.2f}")
print(f"Cash annual return (actual): {cash_annual_return:.2%}")
print(f"Strategy annual return net of cash: {strategy_annual_return_net_of_cash:.2%}")
print(f"Average exposure: {avg_exposure:.2%}")
print(f"Exposure-adjusted annual return net of cash: {exposure_adjusted_annual_return_net_of_cash:.2%}")
print(f"Optimal portfolio annual return: {opt_annual_return:.2%}")
print(f"Equal weight annual return: {eq_annual_return:.2%}")

Final portfolio value: $1,343,843.03
Signal strategy annual return: 6.15%
Total cash interest earned: $37,609.81
Cash annual return (actual): 0.75%
Strategy annual return net of cash: 5.40%
Average exposure: 67.31%
Exposure-adjusted annual return net of cash: 8.02%
Optimal portfolio annual return: 15.81%
Equal weight annual return: 8.58%
